# CLIP Exploration: ST-CLIP Baseline vs FashionCLIP

**Phase 3 Day 1: Vision Embedding Model Selection**

This notebook helps you:
1. Compare ST-CLIP (Sentence-Transformers baseline) vs FashionCLIP (domain-adapted)
2. Visualize image similarities
3. Test text-to-image and image-to-image retrieval
4. **Optimized for Google Colab A100/H100**

**Date**: 2026-02-18  
**Production Model**: FashionCLIP (`patrickjohncyh/fashion-clip`)  
**Baseline**: ST-CLIP ViT-B/32 (`clip-ViT-B-32`)

---

**Important Note**:  
- `clip-ViT-B-32` is a Sentence-Transformers wrapped CLIP model (baseline)
- It is **NOT** the LAION b79k checkpoint
- For production H&M fashion search, we use **FashionCLIP** (domain-adapted)

## Setup & GPU Detection

First, let's check what hardware we're running on.

In [ ]:
# Check GPU availability and specs
import torch
import sys

print("=" * 60)
print("ENVIRONMENT CHECK")
print("=" * 60)
print(f"Python: {sys.version}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA Version: {torch.version.cuda}")
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
    device = "cuda"
elif torch.backends.mps.is_available():
    print("Using MPS (Apple Silicon)")
    device = "mps"
else:
    print("Using CPU")
    device = "cpu"

print(f"\n✅ Selected Device: {device}")
print("=" * 60)

## Install Dependencies (Colab Only)

Run this cell if you're on Google Colab.

In [ ]:
# Uncomment and run if on Google Colab
# !pip install -q sentence-transformers pillow matplotlib numpy

# Optional: Enable HF transfer for faster model downloads
# import os
# os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'

## Import Libraries

In [ ]:
import numpy as np
from PIL import Image
from sentence_transformers import SentenceTransformer
import matplotlib.pyplot as plt
from pathlib import Path
import time

# Enable mixed precision for faster inference on A100/H100
if device == "cuda":
    torch.set_float32_matmul_precision('high')
    print("✅ Mixed precision enabled for faster inference")

print("✅ Libraries imported successfully")

## 1. Load Models

We'll load both models:
- **ST-CLIP Baseline**: `clip-ViT-B-32` (Sentence-Transformers)
- **FashionCLIP (PRODUCTION)**: `patrickjohncyh/fashion-clip` (domain-adapted)

Both output 512-dimensional embeddings.

In [ ]:
# Load ST-CLIP Baseline
print("Loading ST-CLIP ViT-B/32 (baseline)...")
start = time.time()
baseline_model = SentenceTransformer('clip-ViT-B-32', device=device)
baseline_time = time.time() - start
print(f"✅ ST-CLIP loaded in {baseline_time:.2f}s\n")

# Load FashionCLIP (PRODUCTION)
print("Loading FashionCLIP (domain-adapted, PRODUCTION)...")
start = time.time()
fashion_model = SentenceTransformer('patrickjohncyh/fashion-clip', device=device)
fashion_time = time.time() - start
print(f"✅ FashionCLIP loaded in {fashion_time:.2f}s\n")

print("=" * 60)
print("MODELS READY FOR INFERENCE")
print("=" * 60)
print("Baseline: ST-CLIP ViT-B/32")
print("Production: FashionCLIP (patrickjohncyh/fashion-clip)")
print(f"Device: {device}")
print("=" * 60)

## 2. Test Single Image Embedding

Upload a fashion product image or use a local path.

In [ ]:
# TODO: Update path or upload image
test_image_path = "../../data/vision_test/sample.jpg"

# For Colab: Uncomment to upload
# from google.colab import files
# uploaded = files.upload()
# test_image_path = list(uploaded.keys())[0]

# Load and display
img = Image.open(test_image_path).convert("RGB")
plt.figure(figsize=(6, 8))
plt.imshow(img)
plt.axis('off')
plt.title(f"Test Image: {Path(test_image_path).name}")
plt.show()

In [ ]:
# Generate embeddings
print("Generating embeddings...")
with torch.no_grad():
    baseline_emb = baseline_model.encode(img, convert_to_tensor=True, normalize_embeddings=True)
    fashion_emb = fashion_model.encode(img, convert_to_tensor=True, normalize_embeddings=True)

# Convert and ensure (512,) shape
baseline_emb_np = baseline_emb.detach().cpu().numpy().squeeze()
fashion_emb_np = fashion_emb.detach().cpu().numpy().squeeze()

print(f"\nST-CLIP shape: {baseline_emb_np.shape}")
print(f"FashionCLIP shape: {fashion_emb_np.shape}")
print(f"\nST-CLIP L2 norm: {np.linalg.norm(baseline_emb_np):.6f}")
print(f"FashionCLIP L2 norm: {np.linalg.norm(fashion_emb_np):.6f}")

assert baseline_emb_np.shape == (512,)
assert fashion_emb_np.shape == (512,)
print("\n✅ Both embeddings correct: (512,)")

## 3. Text-to-Image Query

**Key evaluation**: Text-image alignment within each model's space.

In [ ]:
queries = [
    "black elegant dress",
    "casual t-shirt",
    "summer outfit",
    "formal business attire",
    "floral print dress"
]

print("=" * 60)
print("TEXT-TO-IMAGE SIMILARITY")
print("=" * 60)

for query in queries:
    with torch.no_grad():
        baseline_text = baseline_model.encode(query, convert_to_tensor=True, normalize_embeddings=True).cpu().numpy().squeeze()
        fashion_text = fashion_model.encode(query, convert_to_tensor=True, normalize_embeddings=True).cpu().numpy().squeeze()
    
    baseline_sim = np.dot(baseline_text, baseline_emb_np)
    fashion_sim = np.dot(fashion_text, fashion_emb_np)
    
    print(f"\nQuery: '{query}'")
    print(f"  ST-CLIP:     {baseline_sim:.4f}")
    print(f"  FashionCLIP: {fashion_sim:.4f}")
    
    if fashion_sim > baseline_sim:
        print(f"  → FashionCLIP better by {(fashion_sim - baseline_sim):.4f}")
    else:
        print(f"  → ST-CLIP better by {(baseline_sim - fashion_sim):.4f}")

print("\n" + "=" * 60)

## Summary

**Production Decision**: FashionCLIP selected for Phase 3

**Next Steps (Day 2)**:
1. Batch process H&M images with FashionCLIP
2. Save to Parquet (image_id, embedding_512d)
3. Prepare for Milvus (Day 3)

---

**Status**: 🟢 Day 1 Complete

# CLIP Exploration: ST-CLIP Baseline vs FashionCLIP

**Phase 3 Day 1: Vision Embedding Model Selection**

This notebook helps you:
1. Compare ST-CLIP (Sentence-Transformers baseline) vs FashionCLIP (domain-adapted)
2. Visualize image similarities
3. Test text-to-image and image-to-image retrieval
4. **Optimized for Google Colab A100/H100**

**Date**: 2026-02-18  
**Production Model**: FashionCLIP (`patrickjohncyh/fashion-clip`)  
**Baseline**: ST-CLIP ViT-B/32 (`clip-ViT-B-32`)

---

**Important Note**: 
- `clip-ViT-B-32` is a Sentence-Transformers wrapped CLIP model (baseline)
- It is **NOT** the LAION b79k checkpoint
- For production H&M fashion search, we use **FashionCLIP** (domain-adapted)

## Setup & GPU Detection

First, let's check what hardware we're running on (especially important for Colab).

In [ ]:
# Check GPU availability and specs
import torch
import sys

print("=" * 60)
print("ENVIRONMENT CHECK")
print("=" * 60)
print(f"Python: {sys.version}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA Version: {torch.version.cuda}")
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
    device = "cuda"
elif torch.backends.mps.is_available():
    print("Using MPS (Apple Silicon)")
    device = "mps"
else:
    print("Using CPU")
    device = "cpu"

print(f"\n✅ Selected Device: {device}")
print("=" * 60)

## Install Dependencies (Colab Only)

Run this cell if you're on Google Colab.

In [ ]:
# Uncomment and run if on Google Colab
# !pip install -q sentence-transformers pillow matplotlib numpy

# Optional: Enable HF transfer for faster model downloads
# import os
# os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'

## Import Libraries

In [ ]:
import numpy as np
from PIL import Image
from sentence_transformers import SentenceTransformer
import matplotlib.pyplot as plt
from pathlib import Path
import time

# Enable mixed precision for faster inference on A100/H100
if device == "cuda":
    torch.set_float32_matmul_precision('high')
    print("✅ Mixed precision enabled for faster inference")

print("✅ Libraries imported successfully")

## 1. Load Models

We'll load both models:
- **ST-CLIP Baseline**: `clip-ViT-B-32` (Sentence-Transformers wrapped baseline)
- **FashionCLIP (PRODUCTION)**: `patrickjohncyh/fashion-clip` (domain-adapted for fashion)

Both output 512-dimensional embeddings.

In [ ]:
# Load ST-CLIP Baseline
print("Loading ST-CLIP ViT-B/32 (baseline)...")
start = time.time()
baseline_model = SentenceTransformer('clip-ViT-B-32', device=device)
baseline_time = time.time() - start
print(f"✅ ST-CLIP loaded in {baseline_time:.2f}s\n")

# Load FashionCLIP (PRODUCTION)
print("Loading FashionCLIP (domain-adapted, PRODUCTION)...")
start = time.time()
fashion_model = SentenceTransformer('patrickjohncyh/fashion-clip', device=device)
fashion_time = time.time() - start
print(f"✅ FashionCLIP loaded in {fashion_time:.2f}s\n")

print("=" * 60)
print("MODELS READY FOR INFERENCE")
print("=" * 60)
print("Baseline: ST-CLIP ViT-B/32")
print("Production: FashionCLIP (patrickjohncyh/fashion-clip)")
print(f"Device: {device}")
print("=" * 60)

## 2. Test Single Image Embedding

Upload a fashion product image or use a local path.

In [ ]:
# TODO: Update this path to your test image
# For Colab: Upload image or mount Google Drive
test_image_path = "../../data/vision_test/sample.jpg"

# For Colab: Uncomment to upload image
# from google.colab import files
# uploaded = files.upload()
# test_image_path = list(uploaded.keys())[0]

# Load and display image
img = Image.open(test_image_path).convert("RGB")
plt.figure(figsize=(6, 8))
plt.imshow(img)
plt.axis('off')
plt.title(f"Test Image: {Path(test_image_path).name}")
plt.show()

In [ ]:
# Generate embeddings with both models
print("Generating embeddings...")
with torch.no_grad():
    baseline_emb = baseline_model.encode(img, convert_to_tensor=True, normalize_embeddings=True)
    fashion_emb = fashion_model.encode(img, convert_to_tensor=True, normalize_embeddings=True)

# Convert to numpy and ensure (512,) shape (not (1, 512))
baseline_emb_np = baseline_emb.detach().cpu().numpy().squeeze()
fashion_emb_np = fashion_emb.detach().cpu().numpy().squeeze()

print(f"\nST-CLIP embedding shape: {baseline_emb_np.shape}")
print(f"FashionCLIP embedding shape: {fashion_emb_np.shape}")
print(f"\nST-CLIP L2 norm: {np.linalg.norm(baseline_emb_np):.6f}")
print(f"FashionCLIP L2 norm: {np.linalg.norm(fashion_emb_np):.6f}")

# Verify dimensions
assert baseline_emb_np.shape == (512,), f"Expected (512,), got {baseline_emb_np.shape}"
assert fashion_emb_np.shape == (512,), f"Expected (512,), got {fashion_emb_np.shape}"
print("\n✅ Both embeddings have correct shape: (512,)")

## 3. Compare Embeddings

Visualize embedding distributions (for numerical sanity check only).

**Note**: These are different embedding spaces, so direct cosine similarity between models is NOT semantically meaningful.

In [ ]:
# Plot embedding distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(baseline_emb_np, bins=50, alpha=0.7, color='blue')
axes[0].set_title('ST-CLIP Baseline Embedding Distribution')
axes[0].set_xlabel('Value')
axes[0].set_ylabel('Frequency')
axes[0].axhline(y=0, color='k', linestyle='-', linewidth=0.5)

axes[1].hist(fashion_emb_np, bins=50, alpha=0.7, color='green')
axes[1].set_title('FashionCLIP Embedding Distribution')
axes[1].set_xlabel('Value')
axes[1].set_ylabel('Frequency')
axes[1].axhline(y=0, color='k', linestyle='-', linewidth=0.5)

plt.tight_layout()
plt.show()

# Statistics comparison
print("\nEmbedding Statistics:")
print("-" * 50)
print(f"{'Metric':<20} {'ST-CLIP':<15} {'FashionCLIP':<15}")
print("-" * 50)
print(f"{'Mean':<20} {baseline_emb_np.mean():.6f}       {fashion_emb_np.mean():.6f}")
print(f"{'Std':<20} {baseline_emb_np.std():.6f}       {fashion_emb_np.std():.6f}")
print(f"{'Min':<20} {baseline_emb_np.min():.6f}       {fashion_emb_np.min():.6f}")
print(f"{'Max':<20} {baseline_emb_np.max():.6f}       {fashion_emb_np.max():.6f}")
print("-" * 50)

## 4. Text-to-Image Query

Test how well each model handles text queries for fashion items.

**This is the key evaluation**: How well do text and image embeddings align within each model's space?

In [ ]:
# Example text queries
queries = [
    "black elegant dress",
    "casual t-shirt",
    "summer outfit",
    "formal business attire",
    "floral print dress"
]

print("=" * 60)
print("TEXT-TO-IMAGE SIMILARITY (Within Same Embedding Space)")
print("=" * 60)

for query in queries:
    with torch.no_grad():
        baseline_text_emb = baseline_model.encode(query, convert_to_tensor=True, normalize_embeddings=True).cpu().numpy().squeeze()
        fashion_text_emb = fashion_model.encode(query, convert_to_tensor=True, normalize_embeddings=True).cpu().numpy().squeeze()
    
    # Compute similarity with test image (within same model's space)
    baseline_sim = np.dot(baseline_text_emb, baseline_emb_np)
    fashion_sim = np.dot(fashion_text_emb, fashion_emb_np)
    
    print(f"\nQuery: '{query}'")
    print(f"  ST-CLIP similarity:     {baseline_sim:.4f}")
    print(f"  FashionCLIP similarity: {fashion_sim:.4f}")
    
    # Highlight which model gives higher similarity
    if fashion_sim > baseline_sim:
        print(f"  → FashionCLIP better by {(fashion_sim - baseline_sim):.4f}")
    else:
        print(f"  → ST-CLIP better by {(baseline_sim - fashion_sim):.4f}")

print("\n" + "=" * 60)

## 5. Multi-Image Similarity Matrix (Optional)

If you have multiple test images, compare how each model clusters them.

In [ ]:
# TODO: Add paths to multiple test images
image_paths = [
    # "../../data/vision_test/dress1.jpg",
    # "../../data/vision_test/dress2.jpg",
    # "../../data/vision_test/shirt.jpg",
]

if image_paths:
    print("Loading images...")
    images = []
    baseline_embeddings = []
    fashion_embeddings = []
    
    for img_path in image_paths:
        img = Image.open(img_path).convert("RGB")
        images.append(img)
        
        with torch.no_grad():
            baseline_emb = baseline_model.encode(img, convert_to_tensor=True, normalize_embeddings=True).cpu().numpy().squeeze()
            fashion_emb = fashion_model.encode(img, convert_to_tensor=True, normalize_embeddings=True).cpu().numpy().squeeze()
        
        baseline_embeddings.append(baseline_emb)
        fashion_embeddings.append(fashion_emb)
    
    # Compute similarity matrices
    n = len(images)
    baseline_sim_matrix = np.zeros((n, n))
    fashion_sim_matrix = np.zeros((n, n))
    
    for i in range(n):
        for j in range(n):
            baseline_sim_matrix[i, j] = np.dot(baseline_embeddings[i], baseline_embeddings[j])
            fashion_sim_matrix[i, j] = np.dot(fashion_embeddings[i], fashion_embeddings[j])
    
    # Plot similarity matrices
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    im1 = axes[0].imshow(baseline_sim_matrix, cmap='viridis', vmin=0, vmax=1)
    axes[0].set_title('ST-CLIP Baseline Similarity Matrix')
    axes[0].set_xlabel('Image Index')
    axes[0].set_ylabel('Image Index')
    plt.colorbar(im1, ax=axes[0])
    
    im2 = axes[1].imshow(fashion_sim_matrix, cmap='viridis', vmin=0, vmax=1)
    axes[1].set_title('FashionCLIP Similarity Matrix')
    axes[1].set_xlabel('Image Index')
    axes[1].set_ylabel('Image Index')
    plt.colorbar(im2, ax=axes[1])
    
    plt.tight_layout()
    plt.show()
else:
    print("📝 Add multiple image paths to test similarity matrix")

## 6. Summary & Decision

### Key Findings:

1. **Both models produce 512-dimensional L2-normalized embeddings** ✅
2. **ST-CLIP (baseline)**: General-purpose, stable, good starting point
3. **FashionCLIP (PRODUCTION)**: Domain-adapted, better for fashion-specific queries

### Production Decision:

**We will use FashionCLIP (`patrickjohncyh/fashion-clip`) for Phase 3** because:
- Trained on fashion data (H&M dataset aligns with this domain)
- Better text-to-image alignment for fashion queries
- Same embedding dimension (512) as baseline
- Compatible with Milvus vector search

### Next Steps (Day 2):
1. Batch process H&M product images with FashionCLIP
2. Save embeddings to Parquet (`image_id, embedding_512d`)
3. Ingest into Milvus image collection (Day 3)

---

### Hardware Performance (record your results):
- **GPU**: _____ (e.g., A100, H100, T4)
- **Model load time**: _____s (baseline) + _____s (fashion)
- **Single image inference**: _____ms
- **Batch (16 images)**: _____ms

---

**Status**: 🟢 Day 1 Complete - FashionCLIP selected for production

# CLIP Exploration: LAION vs FashionCLIP

This notebook helps you:
1. Compare LAION CLIP vs FashionCLIP embeddings
2. Visualize image similarities
3. Test text-to-image and image-to-image retrieval

**Date**: 2026-02-18  
**Phase**: 3 Week 1 Day 1

## Setup

In [ ]:
import sys
sys.path.append('../src')

import numpy as np
import torch
from PIL import Image
from sentence_transformers import SentenceTransformer
import matplotlib.pyplot as plt
from pathlib import Path

# Check device
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")

## 1. Load Models

We'll load both LAION CLIP (baseline) and FashionCLIP (domain-adapted) for comparison.

In [ ]:
# Load LAION CLIP (baseline)
print("Loading LAION CLIP...")
laion_model = SentenceTransformer('clip-ViT-B-32', device=device)
print("✅ LAION CLIP loaded")

# Load FashionCLIP (domain-adapted)
print("\nLoading FashionCLIP...")
fashion_model = SentenceTransformer('patrickjohncyh/fashion-clip', device=device)
print("✅ FashionCLIP loaded")

## 2. Test Single Image Embedding

In [ ]:
# TODO: Update this path to your test image
test_image_path = "../../data/vision_test/sample.jpg"

# Load and display image
img = Image.open(test_image_path).convert("RGB")
plt.figure(figsize=(6, 8))
plt.imshow(img)
plt.axis('off')
plt.title(f"Test Image: {Path(test_image_path).name}")
plt.show()

In [ ]:
# Generate embeddings with both models
with torch.no_grad():
    laion_emb = laion_model.encode(img, convert_to_tensor=True, normalize_embeddings=True)
    fashion_emb = fashion_model.encode(img, convert_to_tensor=True, normalize_embeddings=True)

# Convert to numpy
laion_emb_np = laion_emb.cpu().numpy()
fashion_emb_np = fashion_emb.cpu().numpy()

print(f"LAION CLIP embedding shape: {laion_emb_np.shape}")
print(f"FashionCLIP embedding shape: {fashion_emb_np.shape}")
print(f"\nLAION L2 norm: {np.linalg.norm(laion_emb_np):.6f}")
print(f"FashionCLIP L2 norm: {np.linalg.norm(fashion_emb_np):.6f}")

## 3. Compare Embeddings

Visualize the difference between LAION and FashionCLIP embeddings for the same image.

In [ ]:
# Plot embedding distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(laion_emb_np, bins=50, alpha=0.7, color='blue')
axes[0].set_title('LAION CLIP Embedding Distribution')
axes[0].set_xlabel('Value')
axes[0].set_ylabel('Frequency')

axes[1].hist(fashion_emb_np, bins=50, alpha=0.7, color='green')
axes[1].set_title('FashionCLIP Embedding Distribution')
axes[1].set_xlabel('Value')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

# Compute cosine similarity between the two embeddings
similarity = np.dot(laion_emb_np, fashion_emb_np)
print(f"\nCosine similarity between LAION and FashionCLIP embeddings: {similarity:.4f}")
print("(Higher means models represent this image similarly)")

## 4. Text-to-Image Query (Optional)

Test how well each model handles text queries for fashion items.

In [ ]:
# Example text queries
queries = [
    "black elegant dress",
    "casual t-shirt",
    "summer outfit",
    "formal business attire"
]

# Encode text queries
print("Text Query Embeddings:\n")
for query in queries:
    with torch.no_grad():
        laion_text_emb = laion_model.encode(query, convert_to_tensor=True, normalize_embeddings=True).cpu().numpy()
        fashion_text_emb = fashion_model.encode(query, convert_to_tensor=True, normalize_embeddings=True).cpu().numpy()
    
    # Compute similarity with test image
    laion_sim = np.dot(laion_text_emb, laion_emb_np)
    fashion_sim = np.dot(fashion_text_emb, fashion_emb_np)
    
    print(f"Query: '{query}'")
    print(f"  LAION similarity:   {laion_sim:.4f}")
    print(f"  Fashion similarity: {fashion_sim:.4f}")
    print()

## 5. Multi-Image Similarity Matrix (Optional)

If you have multiple test images, create a similarity matrix.

In [ ]:
# TODO: Add paths to multiple test images
image_paths = [
    # "../data/vision_test/dress1.jpg",
    # "../data/vision_test/dress2.jpg",
    # "../data/vision_test/shirt.jpg",
]

if image_paths:
    # Load and embed images
    images = []
    laion_embeddings = []
    fashion_embeddings = []
    
    for img_path in image_paths:
        img = Image.open(img_path).convert("RGB")
        images.append(img)
        
        with torch.no_grad():
            laion_emb = laion_model.encode(img, convert_to_tensor=True, normalize_embeddings=True).cpu().numpy()
            fashion_emb = fashion_model.encode(img, convert_to_tensor=True, normalize_embeddings=True).cpu().numpy()
        
        laion_embeddings.append(laion_emb)
        fashion_embeddings.append(fashion_emb)
    
    # Compute similarity matrices
    n = len(images)
    laion_sim_matrix = np.zeros((n, n))
    fashion_sim_matrix = np.zeros((n, n))
    
    for i in range(n):
        for j in range(n):
            laion_sim_matrix[i, j] = np.dot(laion_embeddings[i], laion_embeddings[j])
            fashion_sim_matrix[i, j] = np.dot(fashion_embeddings[i], fashion_embeddings[j])
    
    # Plot similarity matrices
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    im1 = axes[0].imshow(laion_sim_matrix, cmap='viridis', vmin=0, vmax=1)
    axes[0].set_title('LAION CLIP Similarity Matrix')
    axes[0].set_xlabel('Image Index')
    axes[0].set_ylabel('Image Index')
    plt.colorbar(im1, ax=axes[0])
    
    im2 = axes[1].imshow(fashion_sim_matrix, cmap='viridis', vmin=0, vmax=1)
    axes[1].set_title('FashionCLIP Similarity Matrix')
    axes[1].set_xlabel('Image Index')
    axes[1].set_ylabel('Image Index')
    plt.colorbar(im2, ax=axes[1])
    
    plt.tight_layout()
    plt.show()
else:
    print("Add multiple image paths to test similarity matrix")

## 6. Summary

**Key Findings:**
- Both models produce 512-dimensional L2-normalized embeddings ✅
- LAION CLIP: General-purpose, stable baseline
- FashionCLIP: Domain-adapted, may perform better for fashion-specific queries

**Next Steps (Day 2):**
- Batch process H&M product images
- Save embeddings to Parquet
- Prepare for Milvus ingestion (Day 3)

---
**Status**: 🟢 Day 1 Complete